In [6]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from pathlib import Path
import pprint
import json
import pandas as pd
from datetime import datetime

ModuleNotFoundError: No module named 'dotenv'

In [ ]:
def campaign_generator():
    client_type = [
        "technology",
        "legal",
        "retail",
        "manufacturing",
        "healthcare",
        "insurance",
        "finance",
        "real estate",
        "telecommunications",
        "hospitality",
        "construction",
        "consulting"
    ]

    # iterate on the prompt a bit.
    original_prompt = """
    You are a sales manager responsible for creating sales campaigns for clients. 
    The client wants to present this campaign to potential investors or customers of the product, 
    but does not want to seem overly eager so as to not scare them away. The audience who will 
    read these campaigns is easily put off by feeling like something is being sold to them.
    The campaign should only be a high level explanation of what the company does, without try
    to woo potential investors or customers. The client will specify what type of industry they are in and would like you to generate a campaign detailing what he sells. 
    Encorporate more specific and unique things about each product beyond basic industry identifying traits. 
    Draft 2-3 sentences for your client given their industry. Do not start with the word introducing. The product can be anything. 
    It does not have to be a software application or service. Here are some examples of what a good campaign looks like. 
    For someone in communications, a good campaign might say: This product is a messaging app for businesses that connects people to the information they need. 
    By bringing people together to work as one unified team, we transform the way organizations communicate. 
    For someone in retail, a good campaign might say: A customer relationship management (CRM) software tailored for small retail businesses to manage customer interactions and sales. 
    For someone in marketing, a good campaign might say: A marketing automation software aimed at marketing managers and directors.
    """
    NUM_CAMPAIGNS = 15
    prompt = """
        # Who you are: 
        You are a sales manager responsible for creating sales campaigns that describe what a client's company sells. 
        The client is does not want the campaigns generated to convey the sentiment that something is being sold to them.
        Draft campaigns that provide a high-level explanation of what products and services the client's company offers,
        without trying to woo potential investors or customers. Be matter of fact and incorporate specific and unique offerings 
        about each product or service. Incorporate information beyond basic industry identifying traits. 

        # Sales Campaigns:
        The client will provide their industry. Each campaign should be at least 3 sentences long. 
        Do not start with the word introducing. Do not seperate the campaigns into description and offering 
        fields. The offering can be any product or service related to the client's industry.  

        # Examples:
        All campaigns should be returned in the following json format:

        "campaigns" : {
            "industry" : [client_industry_0, client_industry_0, client_industry_0],
            "description" : [campaign_description_0, campaign_description_1, ..., campaign_description_n]
        }

        For someone in communications, a good campaign might say: 
            This product is a messaging app for businesses that connects people to the information they need. 
            By bringing people together to work as one unified team, we transform the way organizations communicate.
            
        For someone in retail, a good campaign might say: 
            A customer relationship management (CRM) software tailored for small retail businesses to manage customer interactions and sales.
            
        For someone in marketing, a good campaign might say: 
            A marketing automation software aimed at marketing managers and directors.

        # Task:
        Draft 15 campaigns, each at least 3 sentences long and completely unique from each other. Provide your results in JSON format. 
        Just provide the JSON Object, do not use ``` or "json" to describe the output.
    """


    # connect to openai & run through 4o
    #IMPORTANT: This is how I was able to get the API key from the environment given that .env is in the parent directory because it didn't work otherwise, but I don't want to commit this to git if it means that the key can now be viewed in main.py as a result of load_dotenv(). Conor this is also different than your initial structure because .env is not longer in a folder. I am unsure if that will impact the gitignore now.
    
    env_path = Path(__file__).resolve().parent.parent / '.env'
    
    #env_path = Path("/Users/conorhuh/Desktop/Ouros/SJ/Sales-Juice/.env/.env")
    load_dotenv(dotenv_path=env_path)

    model="gpt-4o-mini"

    client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

    results_list = []
    for industry in client_type:

        completion = client.chat.completions.create(
            model=model,
            messages=[
                #context
                {"role": "system", "content": prompt}, 
                #input, for our purposes the client_type
                {"role": "user", "content": f"The client is in the {industry} industry"}
            ]
        )

        result = completion.choices[0].message.content
        results_dict = json.loads(result)
        print(results_dict)
        for key, campaign_dict in results_dict.items():
            for i in range(NUM_CAMPAIGNS):
                temp_list = [campaign_dict['industry'][i], campaign_dict['description'][i]]
                results_list.append(temp_list)
                
            # for campaign in campaign_dict['description']:
            #     results_list.append(campaign)
        
    current_datetime = datetime.now()
    scrape_time = current_datetime.strftime("%Y.%m.%d")
    test_campaigns = pd.DataFrame(results_list)
    #test_campaigns.to_csv(Path(f"/Users/conorhuh/Desktop/Ouros/SJ/Sales-Juice/data/campaigns/test_campaigns_{scrape_time}.csv").as_posix())
    test_campaigns.to_csv(Path(f"/Users/aidankeeleycain/untitled folder/Sales-Juice/data/test_campaigns_{scrape_time}.csv").as_posix())
